2200018315_Rahman Nendhiarto_Deep Learning

##### Copyright 2019 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

## Tujuan Praktikum ##
Pada praktikum ini mahasiswa akan membangun Sequence to Sequence model, yaitu model yang menerima input berupa data sekuens dan mengeluarkan output berupa data sekuens juga. Aplikasi yang akan dibangun adalah mesin translasi bahasa Spanyol-English.

# Neural machine translation with attention

<table class="tfo-notebook-buttons" align="left">
  <td>
    <a target="_blank" href="https://www.tensorflow.org/text/tutorials/nmt_with_attention">
    <img src="https://www.tensorflow.org/images/tf_logo_32px.png" />
    View on TensorFlow.org</a>
  </td>
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/tensorflow/text/blob/master/docs/tutorials/nmt_with_attention.ipynb">
    <img src="https://www.tensorflow.org/images/colab_logo_32px.png" />
    Run in Google Colab</a>
  </td>
  <td>
    <a target="_blank" href="https://github.com/tensorflow/text/blob/master/docs/tutorials/nmt_with_attention.ipynb">
    <img src="https://www.tensorflow.org/images/GitHub-Mark-32px.png" />
    View source on GitHub</a>
  </td>
  <td>
    <a href="https://storage.googleapis.com/tensorflow_docs/text/docs/tutorials/nmt_with_attention.ipynb"><img src="https://www.tensorflow.org/images/download_logo_32px.png" />Download notebook</a>
  </td>
</table>

Notebook ini mendemonstrasikan bagaimana untuk melatih sebuah model *sequence-to-sequence* (seq2seq)  untuk translasi Spanish-to-English berdasarkan paper [Effective Approaches to Attention-based Neural Machine Translation](https://arxiv.org/abs/1508.04025v5) (Luong et al., 2015).

<table>
<tr>
  <td>
   <img width=400 src="https://www.tensorflow.org/images/tutorials/transformer/RNN%2Battention-words-spa.png"/>
  </td>
</tr>
<tr>
  <th colspan=1>This tutorial: An encoder/decoder connected by attention.</th>
<tr>
</table>

Meskipun aristektur ini cukup *outdated*, project ini masih penting untuk memahami cara kerja *sequence-to-sequence* model dan mekanisme atensi (sebelum mempelajari [Transformers](transformer.ipynb)).



Project ini membutuhkan pengetahuan dasar/fundamental TensorFlow di bawah level Keras layer:
  * [Bekerja dengan tensors](https://www.tensorflow.org/guide/tensor) directly
  * [Menulis model custom `keras.Model`s dan `keras.layers`](https://www.tensorflow.org/guide/keras/custom_layers_and_models)

Setelah melakukan training model, Anda dapat menginputkan kalimat dalam bahasa Spanish, seperti "*¿todavia estan en casa?*", dan mengembalikan hasil translasinya dalam bahasa Inggris: "*are you still at home?*"

The resulting model is exportable as a `tf.saved_model`, so it can be used in other TensorFlow environments.

Gambar di bawah ini menunjukkan bagian-bagian dari kalimat dan bobot attention yang diberikan pada setiap kata (bagian dari kalimat)

<img src="https://tensorflow.org/images/spanish-english.png" alt="spanish-english attention plot">



## Setup

In [ ]:
!pip install "tensorflow-text>=2.11"
!pip install einops

In [ ]:
import numpy as np

import typing
from typing import Any, Tuple

import einops
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import tensorflow as tf
import tensorflow_text as tf_text

Tutorial ini menggunakan banyak API tingkat rendah yang mudah membuat *shape* dari *tensor* yang dihasilkan menjadi salah. Kelas ini digunakan untuk memeriksa *shape* dari tensor di seluruh bagian notebook ini.


In [ ]:
#@title
class ShapeChecker():
  def __init__(self):
    # Keep a cache of every axis-name seen
    self.shapes = {}

  def __call__(self, tensor, names, broadcast=False):
    if not tf.executing_eagerly():
      return

    parsed = einops.parse_shape(tensor, names)

    for name, new_dim in parsed.items():
      old_dim = self.shapes.get(name, None)

      if (broadcast and new_dim == 1):
        continue

      if old_dim is None:
        # If the axis name is new, add its length to the cache.
        self.shapes[name] = new_dim
        continue

      if new_dim != old_dim:
        raise ValueError(f"Shape mismatch for dimension: '{name}'\n"
                         f"    found: {new_dim}\n"
                         f"    expected: {old_dim}\n")

## The data

Tutorial ini menggunakan dataset bahasa yang disediakan oleh [Anki](http://www.manythings.org/anki/). Dataset ini menyimpan data dalam format pasangan kalimat translasi sebagai berikut:

```
May I borrow this book?	¿Puedo tomar prestado este libro?
```

Dalam latihan ini kita hanya akan menggunakan dataset bahasa Spanish-English meskipun pada koleksi dataset Anki terdapat banyak translasi bahasa.

### Download and prepare the dataset

Setelah mendownload dataset, langkah-langkah preprocessing yang dilakukan adalah sebagai berikut:

1. Tambahkan token untuk menandai *start* dan *end* pada setiap kalimat.
2. Bersihkan kalimat dengan menghapus special characters.
3. Buatlah *word index* dan *reverse word index* (dictionaries untuk mapping word → id dan id → word).
4. Lakukan padding pada setiap kalimat sehingga setiap kalimat mencapai panjang maksimal.

In [ ]:
# Download the file
import pathlib

path_to_zip = tf.keras.utils.get_file(
    'spa-eng.zip', origin='http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip',
    extract=True)

path_to_file = pathlib.Path(path_to_zip).parent/'spa-eng_extracted/spa-eng/spa.txt'

In [ ]:
def load_data(path):
  text = path.read_text(encoding='utf-8')

  lines = text.splitlines()
  pairs = [line.split('\t') for line in lines]

  context = np.array([context for target, context in pairs])
  target = np.array([target for target, context in pairs])

  return target, context

In [ ]:
target_raw, context_raw = load_data(path_to_file)
print(context_raw[-1])

In [ ]:
print(target_raw[-1])

### Create a tf.data dataset

Dari arrays of strings yang sudah di-download, kita dapat membuat `tf.data.Dataset` yang dapat melakukan *shuffles* dan menge-set batches dari array tersebut secara efisien:

In [ ]:
BUFFER_SIZE = len(context_raw)
BATCH_SIZE = 64

is_train = np.random.uniform(size=(len(target_raw),)) < 0.8

train_raw = (
    tf.data.Dataset
    .from_tensor_slices((context_raw[is_train], target_raw[is_train]))
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE))
val_raw = (
    tf.data.Dataset
    .from_tensor_slices((context_raw[~is_train], target_raw[~is_train]))
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE))

In [ ]:
for example_context_strings, example_target_strings in train_raw.take(1):
  print(example_context_strings[:5])
  print()
  print(example_target_strings[:5])
  break

### Text preprocessing

Salah satu tujuan dari tutorial ini adalah untuk membangun model yang dapat diekspor sebagai `tf.saved_model`. Agar model yang diekspor tersebut berguna, model tersebut harus mengambil input `tf.string`, dan mengembalikan output `tf.string`: Semua pemrosesan teks terjadi di dalam model. Terutama menggunakan lapisan `layers.TextVectorization`.

#### Standardization

Model ini menangani teks multibahasa dengan kosakata terbatas. Jadi penting untuk menstandardisasi teks masukan.

Langkah pertama adalah normalisasi Unicode untuk memisahkan karakter ber-aksen dan mengganti karakter yang compatible dengan padanan ASCII-nya.

Paket `tensorflow_text` berisi operasi normalisasi unicode:

In [ ]:
example_text = tf.constant('¿Todavía está en casa?')

print(example_text.numpy())
print(tf_text.normalize_utf8(example_text, 'NFKD').numpy())

Normalisasi Unicode akan menjadi langkah pertama pada standarisasi teks:

In [ ]:
def tf_lower_and_split_punct(text):
  # Split accented characters.
  text = tf_text.normalize_utf8(text, 'NFKD')
  text = tf.strings.lower(text)
  # Keep space, a to z, and select punctuation.
  text = tf.strings.regex_replace(text, '[^ a-z.?!,¿]', '')
  # Add spaces around punctuation.
  text = tf.strings.regex_replace(text, '[.?!,¿]', r' \0 ')
  # Strip whitespace.
  text = tf.strings.strip(text)

  text = tf.strings.join(['[START]', text, '[END]'], separator=' ')
  return text

In [ ]:
print(example_text.numpy().decode())
print(tf_lower_and_split_punct(example_text).numpy().decode())

#### Text Vectorization

Fungsi standarisasi ini akan dibungkus dalam lapisan `tf.keras.layers.TextVectorization` yang akan menangani ekstraksi kosakata dan konversi teks input menjadi urutan token.

In [ ]:
max_vocab_size = 5000

context_text_processor = tf.keras.layers.TextVectorization(
    standardize=tf_lower_and_split_punct,
    max_tokens=max_vocab_size,
    ragged=True)

Lapisan `TextVectorization` dan banyak [lapisan praproses Keras](https://www.tensorflow.org/guide/keras/preprocessing_layers) lainnya memiliki metode `adapt`. Metode ini membaca satu periode data pelatihan, dan bekerja seperti `Model.fit`. Metode `adapt` ini menginisialisasi lapisan berdasarkan data. Di sini, metode ini menentukan kosakata:

In [ ]:
context_text_processor.adapt(train_raw.map(lambda context, target: context))

# Here are the first 10 words from the vocabulary:
context_text_processor.get_vocabulary()[:10]

Itu  adalah  `TextVectorization` layer untuk bahasa Spanish, sekarang buat dan `.adapt()` layer serupa untuk bahasa Inggris:

In [ ]:
target_text_processor = tf.keras.layers.TextVectorization(
    standardize=tf_lower_and_split_punct,
    max_tokens=max_vocab_size,
    ragged=True)

target_text_processor.adapt(train_raw.map(lambda context, target: target))
target_text_processor.get_vocabulary()[:10]

Berikutnya layer-layer di bawah ini akan mengubah satu batch string (teks) menjadi satu batch token IDs:

In [ ]:
example_tokens = context_text_processor(example_context_strings)
example_tokens[:3, :]

Method `get_vocabulary` dapat digunakan untuk mengubah token IDs kembali menjadi teks:

In [ ]:
context_vocab = np.array(context_text_processor.get_vocabulary())
tokens = context_vocab[example_tokens[0].numpy()]
' '.join(tokens)

Token ID yang dikembalikan di-padding dengan nol, sehingga dapat diubah menjadi 'mask' dengan mudah:

In [ ]:
plt.subplot(1, 2, 1)
plt.pcolormesh(example_tokens.to_tensor())
plt.title('Token IDs')

plt.subplot(1, 2, 2)
plt.pcolormesh(example_tokens.to_tensor() != 0)
plt.title('Mask')

### Process the dataset



Fungsi `process_text` di bawah ini mengubah `Datasets` yang berisi string-string, menjadi tensor token IDs yang di padding dengan 0. Fungsi ini juga mengubah dari pasangan `(context, target)` ke pasangan `((context, target_in), target_out)` untuk training model dengan `keras.Model.fit`. Keras mengharapkan pasangan `(inputs, labels)`, inputnya adalah `(context, target_in)` dan labelnya adalah `target_out`. Perbedaan antara `target_in` dan `target_out` adalah keduanya digeser satu langkah relatif terhadap satu sama lain, sehingga setiap 'target_out' adalah token berikutnya.

Keterangan:
context = kalimat input  target = kalimat output

In [ ]:
def process_text(context, target):
  context = context_text_processor(context).to_tensor()
  target = target_text_processor(target)
  targ_in = target[:,:-1].to_tensor()
  targ_out = target[:,1:].to_tensor()
  return (context, targ_in), targ_out


train_ds = train_raw.map(process_text, tf.data.AUTOTUNE)
val_ds = val_raw.map(process_text, tf.data.AUTOTUNE)

Berikut ini adalah sekuens pertama untuk setiap *target_in* dan *target_out*, dari batch pertama:

In [ ]:
for (ex_context_tok, ex_tar_in), ex_tar_out in train_ds.take(1):
  print(ex_context_tok[0, :10].numpy())
  print()
  print(ex_tar_in[0, :10].numpy())
  print(ex_tar_out[0, :10].numpy())

## The encoder/decoder
Diagram berikut menunjukkan gambaran umum model tersebut. Pada keduanya, encoder berada di sebelah kiri, decoder berada di sebelah kanan. Pada setiap langkah waktu (timestep), output decoder digabungkan dengan output encoder, untuk memprediksi kata berikutnya.

Aslinya (gambar sebelah kiri) terdapat beberapa koneksi tambahan yang sengaja dihilangkan dari model tutorial ini (gambar sebelah kanan), karena umumnya tidak diperlukan, dan sulit diimplementasikan. Koneksi yang dihilangkan tersebut adalah:

1. Memberikan status dari RNN encoder ke RNN decoder
2. Memberikan output *attention* kembali ke input RNN Decoder.

<table>
<tr>
  <td>
   <img width=500 src="https://www.tensorflow.org/images/seq2seq/attention_mechanism.jpg"/>
  </td>
  <td>
   <img width=380 src="https://www.tensorflow.org/images/tutorials/transformer/RNN+attention.png"/>
  </td>
</tr>
<tr>
  <th colspan=1>The original from <a href=https://arxiv.org/abs/1508.04025v5>Effective Approaches to Attention-based Neural Machine Translation</a></th>
  <th colspan=1>This tutorial's model</th>
<tr>
</table>


Definisikan jumlah neuron pada RNN (baik Encoder maupun Decoder) sebagai berikut:

In [ ]:
UNITS = 256

### The encoder

Tujuan dari Encoder adalah memproses urutan sekuens input menjadi urutan vektor yang berguna bagi Decoder saat mencoba memprediksi output berikutnya untuk setiap langkah waktu (timestep). Karena urutan sekuens input bersifat konstan, tidak ada batasan tentang bagaimana informasi dapat mengalir dalam Encoder, jadi gunakan RNN dua arah (*Bidirectional*) untuk melakukan pemrosesan:

<table>
<tr>
  <td>
   <img width=500 src="https://tensorflow.org/images/tutorials/transformer/RNN-bidirectional.png"/>
  </td>
</tr>
<tr>
  <th>A bidirectional RNN</th>
<tr>
</table>

Encoder:

1. Mengambil list token ID (dari `context_text_processor`).

3. Mencari vektor embedding untuk setiap token (Menggunakan `layers.Embedding`).

4. Memproses vektor embedding menjadi sequence baru (*context vector*) (Menggunakan `layers.GRU` dua arah).

5. Mengembalikan sequence yang diproses. Ini akan diteruskan ke attention head.

In [ ]:
class Encoder(tf.keras.layers.Layer):
  def __init__(self, text_processor, units):
    super(Encoder, self).__init__()
    self.text_processor = text_processor
    self.vocab_size = text_processor.vocabulary_size()
    self.units = units

    # The embedding layer converts tokens to vectors
    self.embedding = tf.keras.layers.Embedding(self.vocab_size, units,
                                               mask_zero=True)

    # The RNN layer processes those vectors sequentially.
    self.rnn = tf.keras.layers.Bidirectional(
        merge_mode='sum',  #hasil dari arah forward dan arah backward dijumlahkan
        layer=tf.keras.layers.GRU(units,
                            # Return the sequence and state
                            return_sequences=True,
                            recurrent_initializer='glorot_uniform'))

  def call(self, x):
    shape_checker = ShapeChecker()
    shape_checker(x, 'batch s')

    # 2. The embedding layer looks up the embedding vector for each token.
    x = self.embedding(x)
    shape_checker(x, 'batch s units')

    # 3. The GRU processes the sequence of embeddings.
    x = self.rnn(x)
    shape_checker(x, 'batch s units')

    # 4. Returns the new sequence of embeddings.
    return x

  def convert_input(self, texts):
    texts = tf.convert_to_tensor(texts)
    if len(texts.shape) == 0:
      texts = tf.convert_to_tensor(texts)[tf.newaxis]
    context = self.text_processor(texts).to_tensor()
    context = self(context)
    return context

Cobalah memanggil Encoder untuk contoh sekuens pada ex_context_tok:

In [ ]:
# Encode the input sequence.
encoder = Encoder(context_text_processor, UNITS)
ex_context = encoder(ex_context_tok)

print(f'Context tokens, shape (batch, s): {ex_context_tok.shape}')
print(f'Encoder output, shape (batch, s, units): {ex_context.shape}')

### The attention layer

Attention Layer memungkinkan Decoder mengakses informasi yang diekstrak oleh Encoder. Layer ini menghitung vektor *context* dari seluruh sekuens input, dan menambahkannya ke output Decoder.

Cara paling sederhana untuk menghitung satu vektor dari sekuens adalah dengan mengambil rata-rata di seluruh sekuens (`layers.GlobalAveragePooling1D`). Attention layer menghitung rata-rata **berbobot** (*weighted average*) di seluruh sekuens input. Di mana bobot dihitung dari kombinasi vektor sekuens input dan "Query".

<table>
<tr>
  <td>
   <img width=500 src="https://www.tensorflow.org/images/tutorials/transformer/CrossAttention-new-full.png"/>
  </td>
</tr>
<tr>
  <th colspan=1>The attention layer</th>
<tr>
</table>

In [ ]:
class CrossAttention(tf.keras.layers.Layer):
  def __init__(self, units, **kwargs):
    super().__init__()
    self.mha = tf.keras.layers.MultiHeadAttention(key_dim=units, num_heads=1, **kwargs)
    self.layernorm = tf.keras.layers.LayerNormalization()
    self.add = tf.keras.layers.Add()

  def call(self, x, context):
    shape_checker = ShapeChecker()

    shape_checker(x, 'batch t units')
    shape_checker(context, 'batch s units')

    attn_output, attn_scores = self.mha(
        query=x,
        value=context,
        return_attention_scores=True)

    shape_checker(x, 'batch t units')
    shape_checker(attn_scores, 'batch heads t s')

    # Cache the attention scores for plotting later.
    attn_scores = tf.reduce_mean(attn_scores, axis=1)
    shape_checker(attn_scores, 'batch t s')
    self.last_attention_weights = attn_scores

    x = self.add([x, attn_output])
    x = self.layernorm(x)

    return x

In [ ]:
attention_layer = CrossAttention(UNITS)

# Attend to the encoded tokens
embed = tf.keras.layers.Embedding(target_text_processor.vocabulary_size(),
                                  output_dim=UNITS, mask_zero=True)
ex_tar_embed = embed(ex_tar_in)

result = attention_layer(ex_tar_embed, ex_context)

print(f'Context sequence, shape (batch, s, units): {ex_context.shape}')
print(f'Target sequence, shape (batch, t, units): {ex_tar_embed.shape}')
print(f'Attention result, shape (batch, t, units): {result.shape}')
print(f'Attention weights, shape (batch, t, s):    {attention_layer.last_attention_weights.shape}')

Jika bobot attention dijumlahkan, totalnya adalah  `1` untuk keseluruhan input.

In [ ]:
attention_layer.last_attention_weights[0].numpy().sum(axis=-1)



Berikut ini attention weight pada sekuens input/ context saat `t=0`:

In [ ]:
attention_weights = attention_layer.last_attention_weights
mask=(ex_context_tok != 0).numpy()

plt.subplot(1, 2, 1)
plt.pcolormesh(mask*attention_weights[:, 0, :])
plt.title('Attention weights')

plt.subplot(1, 2, 2)
plt.pcolormesh(mask)
plt.title('Mask');


Karena diinisialisasi dengan nilai random yang kecil di awal, nilai attention weight semuanya hampir sama yaitu mendekati nilai  `1/(sequence_length)`. Ketika training berlangsung, nilai-nilai attention weight ini akan berubah menjadi lebih tidak seragam.

### The decoder

Tugas Decoder adalah menghasilkan prediksi untuk token berikutnya di setiap posisi (timestep) pada sekuens target.

1. Decoder mencari posisi untuk setiap token dalam sekuens target.

2. Decoder menggunakan RNN untuk memproses sekuens target, dan melacak apa yang telah dihasilkannya pada setiap timestep.

3. Decoder menggunakan output RNN sebagai "query" ke Attention layers, saat menangani output Encoder.

4. Di setiap lokasi dalam output, Decoder memprediksi token berikutnya.

Saat pelatihan, model memprediksi kata berikutnya di setiap posisi kata. Jadi penting bahwa informasi hanya mengalir dalam satu arah melalui model. Decoder menggunakan RNN searah (bukan dua arah) untuk memproses urutan target.

Saat menjalankan inferensi dengan model ini, Decoder menghasilkan satu kata pada satu waktu, dan kata-kata tersebut dimasukkan kembali ke dalam model.

<table>
<tr>
  <td>
   <img width=500 src="https://tensorflow.org/images/tutorials/transformer/RNN.png"/>
  </td>
</tr>
<tr>
  <th>A unidirectional RNN</th>
<tr>
</table>

Berikut ini adalah inisiasi dari `Decoder` class'. Pada saat inisiasi, semua layer yang dibutuhkan dibuat.

In [ ]:
class Decoder(tf.keras.layers.Layer):
  @classmethod
  def add_method(cls, fun):
    setattr(cls, fun.__name__, fun)
    return fun

  def __init__(self, text_processor, units):
    super(Decoder, self).__init__()
    self.text_processor = text_processor
    self.vocab_size = text_processor.vocabulary_size()
    self.word_to_id = tf.keras.layers.StringLookup(
        vocabulary=text_processor.get_vocabulary(),
        mask_token='', oov_token='[UNK]')
    self.id_to_word = tf.keras.layers.StringLookup(
        vocabulary=text_processor.get_vocabulary(),
        mask_token='', oov_token='[UNK]',
        invert=True)
    self.start_token = self.word_to_id('[START]')
    self.end_token = self.word_to_id('[END]')

    self.units = units


    # 1. Embedding layer mengubah token IDs menjadi vector
    self.embedding = tf.keras.layers.Embedding(self.vocab_size,
                                               units, mask_zero=True)

    # 2. RNN GRU digunakan untuk memproses vektor target yang dibangkitkan di setiap timestep.
    self.rnn = tf.keras.layers.GRU(units,
                                   return_sequences=True,
                                   return_state=True,
                                   recurrent_initializer='glorot_uniform')

    # 3. Output dari GRU menjadi query bagi  attention layer.
    self.attention = CrossAttention(units)

    # 4. Dense layer menghasilkan nilai logits bagi setiap token output.
    self.output_layer = tf.keras.layers.Dense(self.vocab_size)

#### Training

Selanjutnya, metode `call`  mengambil 3 arguments:

* `inputs` -  pasangan `context, x`  dimana:
  * `context` - vector context dari output Encoder.
  * `x` - sekuens target.
* `state` - Optional,  output `state` sebelumnya dari Decoder (the internal state of the decoder's RNN). Isikan/Gunakan state pada timestep sebelumnya untuk membangkitkan token pada timestep saat ini.
* `return_state` - [Default: False] - Set nilainya menjadi `True` untuk mengembalikan RNN state.

In [ ]:
@Decoder.add_method
def call(self,
         context, x,
         state=None,
         return_state=False):
  shape_checker = ShapeChecker()
  shape_checker(x, 'batch t')
  shape_checker(context, 'batch s units')

  # 1. Lookup the embeddings
  x = self.embedding(x)
  shape_checker(x, 'batch t units')

  # 2. Process the target sequence.
  if tf.config.list_physical_devices('GPU'):
    if state is not None:
        output = self.rnn(x, initial_state=state)
    else:
        output = self.rnn(x)
  else:
    output = self.rnn(x, initial_state=state)

  x = output [0]
  state = output [1]

  shape_checker(x, 'batch t units')

  # 3. Use the RNN output as the query for the attention over the context.
  x = self.attention(x, context)
  self.last_attention_weights = self.attention.last_attention_weights
  shape_checker(x, 'batch t units')
  shape_checker(self.last_attention_weights, 'batch t s')

  # Step 4. Generate logit predictions for the next token.
  logits = self.output_layer(x)
  shape_checker(logits, 'batch t target_vocab_size')

  if return_state:
    return logits, state
  else:
    return logits

Coba membuat instance Decoder

In [ ]:
decoder = Decoder(target_text_processor, UNITS)

Pada training, Decoder akan digunakan sebagai berikut:

Diberikan sekuens input dan sekuens target, untuk setiap token pada sekuens  target Decoder memprediksi token selanjutnya.

In [ ]:
logits = decoder(ex_context, ex_tar_in)

print(f'encoder output shape: (batch, s, units) {ex_context.shape}')
print(f'input target tokens shape: (batch, t) {ex_tar_in.shape}')
print(f'logits shape shape: (batch, target_vocabulary_size) {logits.shape}')

#### Inference

Untuk melakukan inferens, beberapa metode berikut ini dibutuhkan:

In [ ]:
@Decoder.add_method
def get_initial_state(self, context):
  batch_size = tf.shape(context)[0]
  start_tokens = tf.fill([batch_size, 1], self.start_token)
  done = tf.zeros([batch_size, 1], dtype=tf.bool)
  embedded = self.embedding(start_tokens)

  state = self.rnn.get_initial_state(batch_size)[0]
  # atau
  # embedded_size = tf.shape(embedded)[0]
  # state = self.rnn.get_initial_state(embedded_size)[0]

  return start_tokens, done, state

In [ ]:
@Decoder.add_method
def tokens_to_text(self, tokens):
  words = self.id_to_word(tokens)
  result = tf.strings.reduce_join(words, axis=-1, separator=' ')
  result = tf.strings.regex_replace(result, '^ *\[START\] *', '')
  result = tf.strings.regex_replace(result, ' *\[END\] *$', '')
  return result

In [ ]:
@Decoder.add_method
def get_next_token(self, context, next_token, done, state, temperature = 0.0):
  # print("state shape : ", tf.shape(state))
  logits, state = self(
    context, next_token,
    state = state,
    return_state=True)

  # Periksa apakah GPU tersedia
  if tf.config.list_physical_devices('GPU'):
    # print("GPU tersedia. Menggunakan GPU.")
    expected_shape = tf.stack([tf.shape(next_token)[0], self.units])
    if tf.shape(state)[0] != tf.shape(next_token)[0]: # Jika batch dim hilang
        if tf.shape(state)[0] == self.units: # Dan shape-nya hanya (units,)
            # Reshape menjadi (1, units) dan kemudian broadcast ke (batch_size, units)
            state = tf.reshape(state, [1, self.units])
            state = tf.broadcast_to(state, expected_shape)
            # print("state shape : ", tf.shape(state))

  if temperature == 0.0:
    next_token = tf.argmax(logits, axis=-1)
  else:
    logits = logits[:, -1, :]/temperature
    next_token = tf.random.categorical(logits, num_samples=1)

  # If a sequence produces an `end_token`, set it `done`
  done = done | (next_token == self.end_token)
  # Once a sequence is done it only produces 0-padding.
  next_token = tf.where(done, tf.constant(0, dtype=tf.int64), next_token)

  return next_token, done, state

Dengan beberapa fungsi tambahan, Loop untuk pembangkitan token adalah sebagai berikut:

In [ ]:
# Setup the loop variables.
next_token, done, state = decoder.get_initial_state(ex_context)
tokens = []

for n in range(10):
  # print(n, state)
  # Run one step.
  next_token, done, state = decoder.get_next_token(
      ex_context, next_token, done, state, temperature=1.0)
  # print(n, state)
  # Add the token to the output.
  tokens.append(next_token)

# Stack all the tokens together.
tokens = tf.concat(tokens, axis=-1) # (batch, t)

# Convert the tokens back to a a string
result = decoder.tokens_to_text(tokens)
result[:3].numpy()

Karena modelnya tidak di training, model mengeluarkan item dari kosakata hampir secara seragam dan acak.

## The model

Saat ini karena kita sudah memiliki semua komponen, maka kita dapat kombinasikan untuk membangun model untuk proses training:

In [ ]:
class Translator(tf.keras.Model):
  @classmethod
  def add_method(cls, fun):
    setattr(cls, fun.__name__, fun)
    return fun

  def __init__(self, units,
               context_text_processor,
               target_text_processor):
    super().__init__()
    # Build the encoder and decoder
    encoder = Encoder(context_text_processor, units)
    decoder = Decoder(target_text_processor, units)

    self.encoder = encoder
    self.decoder = decoder

  def call(self, inputs):
    context, x = inputs
    context = self.encoder(context)
    logits = self.decoder(context, x)

    #TODO(b/250038731): remove this
    try:
      # Delete the keras mask, so keras doesn't scale the loss+accuracy.
      del logits._keras_mask
    except AttributeError:
      pass

    return logits

Selama training, model akan digunakan sebagai berikut:

In [ ]:
model = Translator(UNITS, context_text_processor, target_text_processor)

logits = model((ex_context_tok, ex_tar_in))

print(f'Context tokens, shape: (batch, s, units) {ex_context_tok.shape}')
print(f'Target tokens, shape: (batch, t) {ex_tar_in.shape}')
print(f'logits, shape: (batch, t, target_vocabulary_size) {logits.shape}')

### Train

Untuk training, kita terapkan fungsi loss dan akurasi:

In [ ]:
def masked_loss(y_true, y_pred):
    # Calculate the loss for each item in the batch.
    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
        from_logits=True, reduction='none')
    loss = loss_fn(y_true, y_pred)

    # Mask off the losses on padding.
    mask = tf.cast(y_true != 0, loss.dtype)
    loss *= mask

    # Return the total.
    return tf.reduce_sum(loss)/tf.reduce_sum(mask)

In [ ]:
def masked_acc(y_true, y_pred):
    # Calculate the loss for each item in the batch.
    y_pred = tf.argmax(y_pred, axis=-1)
    y_pred = tf.cast(y_pred, y_true.dtype)

    match = tf.cast(y_true == y_pred, tf.float32)
    mask = tf.cast(y_true != 0, tf.float32)

    return tf.reduce_sum(match)/tf.reduce_sum(mask)

Konfigurasikan model untuk training:

In [ ]:
model.compile(optimizer='adam',
              loss=masked_loss,
              metrics=[masked_acc, masked_loss])

Model diinisialisasi dengan random, dan harus memberikan probabilitas output yang hampir seragam. Jadi mudah untuk memprediksi nilai awal metrik yang seharusnya:

In [ ]:
vocab_size = 1.0 * target_text_processor.vocabulary_size()

{"expected_loss": tf.math.log(vocab_size).numpy(),
 "expected_acc": 1/vocab_size}

Evaluasi model

In [ ]:
model.evaluate(val_ds, steps=20, return_dict=True)

In [ ]:
history = model.fit(
    train_ds.repeat(),
    epochs=100,
    steps_per_epoch = 100,
    validation_data=val_ds,
    validation_steps = 20,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=3)])

In [ ]:
plt.plot(history.history['loss'], label='loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.ylim([0, max(plt.ylim())])
plt.xlabel('Epoch #')
plt.ylabel('CE/token')
plt.legend()

In [ ]:
plt.plot(history.history['masked_acc'], label='accuracy')
plt.plot(history.history['val_masked_acc'], label='val_accuracy')
plt.ylim([0, max(plt.ylim())])
plt.xlabel('Epoch #')
plt.ylabel('CE/token')
plt.legend()

### Translate

Sekarang setelah model berhasil di-training, implementasikan sebuah fungsi untuk eksekusi secara full translasi `text => text`. Code di bawah ini pada dasarnya sama dengan kode inferens [inference example](#inference) pada Decoder [decoder section](#the_decoder), tetapi juga menampilkan attention weights.

In [ ]:
#@title
@Translator.add_method
def translate(self,
              texts, *,
              max_length=50,
              temperature=0.0):
  # Process the input texts
  context = self.encoder.convert_input(texts)
  batch_size = tf.shape(texts)[0]

  # Setup the loop inputs
  tokens = []
  attention_weights = []
  next_token, done, state = self.decoder.get_initial_state(context)

  for _ in range(max_length):
    # Generate the next token
    next_token, done, state = self.decoder.get_next_token(
        context, next_token, done,  state, temperature)

    # Collect the generated tokens
    tokens.append(next_token)
    attention_weights.append(self.decoder.last_attention_weights)

    if tf.executing_eagerly() and tf.reduce_all(done):
      break

  # Stack the lists of tokens and attention weights.
  tokens = tf.concat(tokens, axis=-1)   # t*[(batch 1)] -> (batch, t)
  self.last_attention_weights = tf.concat(attention_weights, axis=1)  # t*[(batch 1 s)] -> (batch, t s)

  result = self.decoder.tokens_to_text(tokens)
  return result

Berikut ini adalah dua metode yang digunakan pada langkah di atas untuk mengubah input token menjadi teks bahasa target dan mendapatkan kata selanjutnya :

In [ ]:
result = model.translate(['¿Todavía está en casa?']) # Are you still home
result[0].numpy().decode()

Gunakan metode di atas untuk mendapatkan gambar attention plot:

In [ ]:
#@title
@Translator.add_method
def plot_attention(self, text, **kwargs):
  assert isinstance(text, str)
  output = self.translate([text], **kwargs)
  output = output[0].numpy().decode()

  attention = self.last_attention_weights[0]

  context = tf_lower_and_split_punct(text)
  context = context.numpy().decode().split()

  output = tf_lower_and_split_punct(output)
  output = output.numpy().decode().split()[1:]

  fig = plt.figure(figsize=(10, 10))
  ax = fig.add_subplot(1, 1, 1)

  ax.matshow(attention, cmap='viridis', vmin=0.0)

  fontdict = {'fontsize': 14}

  ax.set_xticklabels([''] + context, fontdict=fontdict, rotation=90)
  ax.set_yticklabels([''] + output, fontdict=fontdict)

  ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
  ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

  ax.set_xlabel('Input text')
  ax.set_ylabel('Output text')

In [ ]:
model.plot_attention('¿Todavía está en casa?') # Are you still home

Translate beberapa kalimat lagi dan plot kan hasilnya:

In [ ]:
%%time
# This is my life.
model.plot_attention('Esta es mi vida.')

In [ ]:
%%time
 # Try to find out.'
model.plot_attention('Tratar de descubrir.')

Mesin translasi ini bekerja baik untuk kalimat yang pendek pada umumnya, namun jika kalimat inputnya panjang, model translasi ini kehilangan fokus dan berhenti memberikan hasil yang reasonable. Hal ini didasari oleh dua alasan:

1. Model ini dilatih dengan teacher-forcing untuk memasukkan prediksi kata yang benar di setiap langkah, terlepas dari apapun hasil prediksi model. Model dapat dibuat lebih tangguh jika terkadang diberi hasil prediksinya sendiri.
2. Model memilik akses terhadap output sebelumnya melalui 'state' pada RNN. Jika 'state' RNN kehilangan jejak di mana ia berada dalam sekuens token/kata input, tidak ada cara bagi model untuk memulihkannya. [Transformers](transformer.ipynb) mengatasi masalah ini dengan membuat decoder dapat melihat apa yang telah dioutputkan sejauh ini.

Input data diurutkan berdasarkan panjangnya, lakukan translasi pada kalimat terpanjang:

In [ ]:
long_text = context_raw[-1]

import textwrap
print('Expected output:\n', '\n'.join(textwrap.wrap(target_raw[-1])))

In [ ]:
model.plot_attention(long_text)

FUngsi `translate` bekerja pada batches, sehingga jika memiliki input lebih dari satu kalimat, lebih efisien jika mentranslate sekaligus daripada satu per satu:

In [ ]:
inputs = [
    'Hace mucho frio aqui.', # "It's really cold here."
    'Esta es mi vida.', # "This is my life."
    'Su cuarto es un desastre.' # "His room is a mess"
]

In [ ]:
%%time
for t in inputs:
  print(model.translate([t])[0].numpy().decode())

print()

In [ ]:
%%time
result = model.translate(inputs)

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

Selanjutnya kita akan Export Model

### Export

Untuk mengekspor model, perlu membungkus fungsi `translate` ke dalam `tf.function`. Implementasinya sebagai berikut:


In [ ]:
class Export(tf.Module):
  def __init__(self, model):
    self.model = model

  @tf.function(input_signature=[tf.TensorSpec(dtype=tf.string, shape=[None])])
  def translate(self, inputs):
    return self.model.translate(inputs)

In [ ]:
export = Export(model)

Jalankan `tf.function` untuk kompilasi:

In [ ]:
%%time
_ = export.translate(tf.constant(inputs))

In [ ]:
%%time
result = export.translate(tf.constant(inputs))

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

Sekarang fungsi translate sudah ter-trace dan dapat diekspor dengan `saved_model.save`:

In [ ]:
%%time
tf.saved_model.save(export, 'translator',
                    signatures={'serving_default': export.translate})

In [ ]:
%%time
reloaded = tf.saved_model.load('translator')
_ = reloaded.translate(tf.constant(inputs)) #warmup

In [ ]:
%%time
result = reloaded.translate(tf.constant(inputs))

print(result[0].numpy().decode())
print(result[1].numpy().decode())
print(result[2].numpy().decode())
print()

## Post Test ##

Setelah Anda melakukan eksperimen membuat model Translation Machine dengan Attention Mechanism, jawablah pertanyaan-pertanyaan berikut ini:

1. Metode apa yang dipakai sebagai dasar arsitektur Encoder dan Decoder?
2. Apa fungsi Attention Mechanism?
3. Mengapa padding diperlukan pada data input?
3. (Optional) Jika Attention mechanism dihilangkan, bagaimana hasilnya? Bandingkan jika memakai attention dan tidak memakai attention.


JAWABAN POSTEST

1. Metode apa yang dipakai sebagai dasar  arsitektur Encoder dan Decoder?

  Arsitektur dasar untuk Encoder dan Decoder dalam notebook ini menggunakan Recurrent Neural Networks (RNN), khususnya lapisan GRU (Gated Recurrent Unit). Encoder menggunakan GRU dua arah (Bidirectional) untuk memproses urutan input, sementara Decoder menggunakan GRU satu arah (unidirectional) untuk menghasilkan urutan output.

2. Apa fungsi Attention Mechanism?

  Fungsi utama dari Attention Mechanism di sini adalah memungkinkan Decoder untuk fokus pada bagian-bagian yang relevan dari sekuens input (output dari Encoder) saat memprediksi token berikutnya dalam sekuens output. Ini membantu model menangani kalimat yang lebih panjang dan memastikan bahwa terjemahan memperhatikan semua bagian penting dari kalimat sumber, tidak hanya bagian terakhir yang diproses oleh Encoder.

3. Mengapa padding diperlukan pada data input?

 Padding diperlukan karena kalimat dalam dataset memiliki panjang yang berbeda-beda. Saat memproses data dalam batch (kelompok kalimat), semua kalimat dalam satu batch harus memiliki panjang yang sama agar dapat direpresentasikan sebagai tensor dengan bentuk yang konsisten. Padding menambahkan token khusus (biasanya 0) ke akhir kalimat yang lebih pendek hingga mencapai panjang maksimum dalam batch tersebut. Masking kemudian digunakan untuk mengabaikan token padding ini selama perhitungan loss dan akurasi, sehingga tidak mempengaruhi proses pelatihan.

4. (Optional) Jika Attention mechanism dihilangkan, bagaimana hasilnya?

  Bandingkan jika memakai attention dan tidak memakai attention. Jika Attention Mechanism dihilangkan, model hanya akan mengandalkan "state" terakhir dari RNN Encoder untuk melakukan terjemahan. Untuk kalimat pendek, ini mungkin masih memberikan hasil yang lumayan. Namun, untuk kalimat yang lebih panjang, RNN tradisional cenderung mengalami masalah "vanishing gradient" dan kesulitan mengingat informasi dari awal sekuens input (informasi yang jauh). Akibatnya, model tanpa attention akan kesulitan menghasilkan terjemahan yang akurat dan koheren untuk kalimat panjang, seringkali kehilangan fokus atau menghasilkan terjemahan yang tidak lengkap. Model dengan attention, seperti yang ditunjukkan dalam notebook, dapat melihat kembali seluruh sekuens input dan memberikan bobot yang berbeda pada kata-kata yang paling relevan pada setiap langkah terjemahan, menghasilkan terjemahan yang jauh lebih baik, terutama untuk kalimat yang kompleks atau panjang.

Penjelasan Singkat Terkait Neural Machine Translation dengan Attention-RevisedVersion

Penelitian ini fokus pada pembuatan penerjemah Spanyol-Inggris otomatis menggunakan deep learning. Intinya, kita melatih model komputer untuk "membaca" kalimat Spanyol lalu "menulis" terjemahannya dalam bahasa Inggris.

Kuncinya ada di mekanisme "Attention". Fitur ini membuat model bisa fokus pada bagian penting kalimat Spanyol saat menerjemahkan setiap kata ke Inggris, tidak cuma mengingat garis besar kalimatnya.

Prosesnya dimulai dengan menyiapkan data kalimat Spanyol-Inggris, lalu melatih model menggunakan data tersebut. Model belajar memprediksi kata terjemahan langkah demi langkah.

Setelah dilatih, model ini siap menerjemahkan kalimat baru. Kita bahkan bisa melihat visualisasi "perhatian" model, menunjukkan kata mana yang paling relevan saat menerjemahkan. Tujuannya jelas untuk terjemahan yang lebih akurat, khususnya untuk kalimat yang kompleks.

Singkatnya, ini adalah gambaran bagaimana deep learning dengan attention bekerja untuk terjemahan bahasa.